# diagonal-via-strides — faded example 2: k-th super-diagonal via NumPy as_strided with byte offset

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `diagonal-via-strides`. Running the beacon reports progress on the `Numpy: Diagonal via strides` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Diagonal via strides` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`diagonal-via-strides`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "diagonal-via-strides"
DD_SUBTOPIC = "Numpy: Diagonal via strides"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The k-th super-diagonal `[a[0,k], a[1,k+1], ...]` walks with stride `N + 1` elements but begins at `a[0, k]`, which is `k` elements (i.e. `k * itemsize` bytes) into storage. With `np.lib.stride_tricks.as_strided`, you shift the start by slicing the buffer or, more simply, by building a view whose first element is `a` offset by `k` elements. Its length is `N - k`.

## Faded exercise 2

### Faded — k-th super-diagonal with NumPy as_strided

Complete `numpy_kth_super_diagonal(a, k)`. Given a C-contiguous `(N, N)` array `a` and `0 <= k < N`, return a length-`(N - k)` no-copy view aliasing the k-th super-diagonal `[a[0,k], a[1,k+1], ..., a[N-1-k, N-1]]`.

The stride and shape are provided. You must build `start`, the sub-array whose first element is the diagonal's starting element `a[0, k]`, so that `as_strided` begins there. (Hint: `a.ravel()[k:]` exposes a flat view starting at linear index `k`, which is `a[0, k]`.)

**Fill in:** Computes the flat starting view of the buffer beginning at linear index k (i.e. element a[0, k]), so the strided walk starts on the super-diagonal.

In [ ]:
def numpy_kth_super_diagonal(a, k):
    N = a.shape[0]
    start = a.ravel()[k:]
    return np.lib.stride_tricks.as_strided(
        start, shape=(N - k,), strides=((N + 1) * a.itemsize,)
    )


np.random.seed(0)
N = 5
k = 2
a = np.arange(N * N).reshape(N, N)
d = numpy_kth_super_diagonal(a, k)
print("super-diagonal:", d.tolist())


def _test():
    np.random.seed(0)
    for N in (4, 6, 7):
        a = np.arange(N * N).reshape(N, N).astype(np.int64)
        for k in range(N):
            d = numpy_kth_super_diagonal(a, k)
            ref = np.diagonal(a, offset=k).copy()
            assert d.shape == (N - k,), (N, k, d.shape)
            assert np.array_equal(d, ref), (N, k, d, ref)
            assert np.shares_memory(d, a)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def numpy_kth_super_diagonal(a, k):
    N = a.shape[0]
    start = a.ravel()[k:]
    return np.lib.stride_tricks.as_strided(
        start, shape=(N - k,), strides=((N + 1) * a.itemsize,)
    )


np.random.seed(0)
N = 5
k = 2
a = np.arange(N * N).reshape(N, N)
d = numpy_kth_super_diagonal(a, k)
print("super-diagonal:", d.tolist())
```
</details>